# Search Metric

source: https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv

In [ ]:
import sys
import os
import json
import pandas as pd
from rich import print
from minsearch import Index


parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)
from evaluation_utils import compute_relevance_total
from ingest import (load_faq_data,
                    build_index,
                    text_search
) 

# load the documents, fillter llm_zoocamp ONLY since the ground_truth data is generated from the llm_zoomcamp ONLY for simplicity
documents = load_faq_data()
documents_llm = [doc for doc in documents if doc['course'] == "llm-zoomcamp"]
print(f'The number of documents = {len(documents_llm)}')

# get the created ground-truth data
df_ground_truth = pd.read_csv('./data/ground_truth_FAQ_data.csv')
ground_truth = df_ground_truth.to_dict(orient="records")
# convert the df to dictionary

# note some answers do not have 5 questions
print(f'The number of ground_truth = {len(ground_truth)}')

The number of documents = 103

The number of ground_truth = 510

In [ ]:
# create search index and fit with document
search_index = build_index(documents_llm) # fit with the documents

# 1. Bind your fitted index to the search function using a lambda
bound_text_search = lambda query: text_search(query=query, search_index=search_index)


# create relavent matrix using text_search
relavent_matrix = compute_relevance_total(ground_truth, bound_text_search)

# Hit Rate

It tells us whether the correct document appears at all.

In [52]:
# Hit Rate (when there is 1 in a line, means the search_index is able to fetch the documents)

def hit_rate(relavent_matrix, num_results=5):
    """Hit Rate (when there is 1 in a line, means the search_index is able to fetch the documents)
        so the hit rate is the percentage of fetching documents
    """
    hit_rate = sum([sum(line) for line in relavent_matrix])/len(relavent_matrix)
    # Note: some lines do not have 5 
    # relavent_matrix = [row + [0] * (num_results - len(row)) for row in relavent_matrix]
    # relavent_matrix = np.array(relavent_matrix)
    # hit_rate = np.sum(relavent_matrix)/len(relavent_matrix)

    # if we use 
    return (hit_rate)

hit_rate_search_index = hit_rate(relavent_matrix)
print(f'hit_rate_search_index={hit_rate_search_index}')

hit_rate_search_index=0.7509803921568627

# Mean Reciprocal Rank (MRR)

It tells us whether if the document appears near the top. A document near the top is most likely used by RAG.

In [ ]:
import numpy as np

def mrr_numpy(relavent_matrix, num_results=5):
    relavent_matrix = [row + [0] * (num_results - len(row)) for row in relavent_matrix]
    arr = np.asarray(relavent_matrix)
    
    # Find the column index of the first '1' for each row. 
    # If no '1' is found, argmax returns 0, so we must mask them out.
    has_hit = (arr == 1).any(axis=1)
    first_hit_ranks = (arr == 1).argmax(axis=1) + 1
    
    # Calculate reciprocal rank only where a hit exists, otherwise 0
    reciprocal_ranks = np.where(has_hit, 1.0 / first_hit_ranks, 0.0)
    
    return reciprocal_ranks.mean()

mrr_numpy(relavent_matrix)


def mean_reciprocal_rank(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

mean_reciprocal_rank(relavent_matrix)

0.6142156862745095

In [59]:
mrr_numpy(relavent_matrix)

np.float64(0.6142156862745098)

# Put Mettric Together

In [61]:
def evaluate(ground_truth, search_function):
    relavent_matrix  = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relavent_matrix ),
        "mean_reciprocal_rank": mean_reciprocal_rank(relavent_matrix ),
    }

# get the created ground-truth data
df_ground_truth = pd.read_csv('./data/ground_truth_FAQ_data.csv')
ground_truth = df_ground_truth.to_dict(orient="records")


# create search index and fit with document
search_index = build_index(documents_llm)
# 1. Bind your fitted index to the search function using a lambda
bound_text_search = lambda query: text_search(query=query, search_index=search_index)

evaluate(ground_truth, bound_text_search)

{'hit_rate': 0.7509803921568627, 'mean_reciprocal_rank': 0.6142156862745095}